
# A full burst pipeline: simulate, search, select, resolve

Everything else in this gallery hands the HMM a prepared photon list. This
example runs the pipeline a real measurement goes through — diffusing molecules
through a confocal volume, a TTTR file, a burst search, burst selection, and
only then the fit — and does it in the regime where burst-wise analysis
*cannot* answer the question.

Three states exchange **faster than a molecule crosses the focus**. Each burst
therefore samples all three and reports their average, so the FRET histogram
shows a single narrow peak where three populations exist. That is not a
shortcoming of the histogram; it is what averaging does. Recovering the states
requires modelling photons individually, which is what :class:`tttrlib.HMM`
does.

The pipeline::

    SimEngine  ->  TTTR  ->  BurstFilter  ->  burst selection  ->  HMM

<div class="alert alert-info"><h4>Note</h4><p>One simplification, stated because it is invisible otherwise: a
   :class:`SimSpecies` carries a single decay shared by both detection channels,
   so the acceptor micro-times here inherit the donor's decay rather than
   carrying the rise a sensitised acceptor really has. The fit uses the same
   shape, so the model is not misspecified against *this* data — but a real
   acceptor decay needs a separate species per emitting pathway.</p></div>


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import tttrlib

STATE_COLORS = ["#4e79a7", "#59a14f", "#e15759"]

E_STATES = (0.25, 0.50, 0.75)
TAU_D0 = 4.0            # unquenched donor lifetime, ns
N_TAC, TAC_NS = 2048, 0.008
K_EXCHANGE = 100.0      # per time unit -- fast compared with the transit
BRIGHTNESS = 4000.0


def _vd(x):
    return tttrlib.VectorDouble([float(v) for v in x])

## 1. Simulate diffusing molecules
Three species are three conformational states, interconverting through the
spontaneous rate matrix ``k_nrad``. Each carries its own brightness split
(which sets $E$) and its own quenched donor decay — so a state differs
in *both* channels, which is the coupling the fit later exploits.



In [ ]:
system = tttrlib.SimSystem()
for E in E_STATES:
    sp = tttrlib.SimSpecies()
    sp.D = 0.5
    sp.q = _vd([BRIGHTNESS * (1 - E), BRIGHTNESS * E])
    t_ns = np.arange(256) * 0.0625
    sp.decay = tttrlib.SimDecay.from_pattern(
        _vd(np.exp(-t_ns / (TAU_D0 * (1 - E)))), 0.0625, 0.0)
    system.add_species(sp)

k = np.full((3, 3), K_EXCHANGE)
np.fill_diagonal(k, 0.0)
system.set_rate_matrices(_vd(np.zeros(9)), _vd(k.ravel()))
system.set_background(_vd([0.0, 0.0]))
system.set_box(2.0, 4.0)
for i in range(3):
    system.set_population(i, 0.25)          # low: single molecules in the volume

excitation = tttrlib.SimGrid.gaussian3d(0.6, 2.0, 2.0, 4.0, 0.05, 1.0)
integrator = tttrlib.SimIntegrator()
integrator.dt = 1e-3
integrator.n_channels = 2
integrator.n_ph_max = 200000
integrator.max_windows = 3_000_000

engine = tttrlib.SimEngine(system, excitation, tttrlib.VectorSimGrid([]), integrator)
engine.run()
print(f"simulated {engine.n_photons()} photons over {engine.current_window()} windows")

## 2. Into a TTTR
The engine emits the same quantities a file holds — macro time, TAC channel,
routing channel — so the rest of the pipeline is the ordinary one and nothing
below this point knows the data were simulated.



In [ ]:
macro = np.asarray(engine.macro_window(), dtype=np.uint64)
micro = np.asarray(engine.micro_time(), dtype=np.uint16)
channel = np.asarray(engine.channel(), dtype=np.int8)

data = tttrlib.TTTR()
data.append_events(macro, micro, channel,
                   np.zeros(len(macro), dtype=np.int8), False, 0)
header = data.get_header()
header.set_number_of_micro_time_channels(N_TAC)
header.set_micro_time_resolution(TAC_NS * 1e-9)
header.set_macro_time_resolution(integrator.dt)

## 3. Burst search



In [ ]:
bf = tttrlib.BurstFilter(data)
bf.set_burst_parameters(min_photons=40, window_photons=10, window_time_max=5e-3)
bf.find_bursts()
bursts = np.asarray(bf.get_bursts()).reshape(-1, 2)
print(f"burst search found {len(bursts)} bursts")

## 4. Burst selection
Selection is not cosmetic. A burst holding two molecules is a superposition of
two chains, and a single-chain model can only explain it as rapid switching —
so coincidence manufactures exactly the signal being looked for. Coincident
bursts are longer *and* hold more photons, and photon count is the better
discriminator of the two.



In [ ]:
sizes = bursts[:, 1] - bursts[:, 0] + 1
keep = (sizes >= 60) & (sizes <= np.quantile(sizes, 0.95))
selected = bursts[keep]
print(f"selection kept {keep.sum()} of {len(bursts)} bursts "
      f"({sizes[keep].min()}-{sizes[keep].max()} photons)")

## 5. The burst-wise view — one peak where three states are



In [ ]:
acceptor = np.asarray(data.routing_channels) == 1
E_burst = np.array([acceptor[s:e + 1].mean() for s, e in selected])
print(f"burst-wise E: mean {E_burst.mean():.3f}, sd {E_burst.std():.3f}   "
      f"(three states at {E_STATES})")

## 6. The photon-wise view
Same bursts, same photons, loaded on the **product alphabet** so each photon
carries its micro-time as well as its channel.



In [ ]:
green = tttrlib.Channel("green"); green.add_component(0, 0, 65535)
red = tttrlib.Channel("red");     red.add_component(1, 0, 65535)

N_BINS = 64
eng = tttrlib.HMM()
eng.set_bursts_from_tttr(data, selected.astype(np.int64), [green, red],
                         min_photons=40, time_scale=1, n_micro_bins=N_BINS)
print(f"HMM sees {eng.get_n_photons()} photons in {eng.get_n_bursts()} bursts, "
      f"{eng.get_n_symbols()} symbols")

dt_ns = eng.get_micro_time_bin_width_ns()
spec = tttrlib.HmmEmissionSpec.uniform(3, 2, N_BINS, dt_ns, TAU_D0)
for state, E in enumerate((0.4, 0.5, 0.6)):          # deliberately clustered start
    spec.set_stream_probability(state, 0, 1 - E)
    spec.set_stream_probability(state, 1, E)
    tau = TAU_D0 * (1 - E)
    spec.set_spectrum(state, 0, tttrlib.HmmLifetimeSpectrum(tau))
    spec.set_spectrum(state, 1, tttrlib.HmmLifetimeSpectrum(tau))

init = tttrlib.HmmModel([1 / 3] * 3, list((np.eye(3) * 0.98 + 0.01).ravel()),
                        spec.build())
init.n_micro_bins = N_BINS
fit = eng.optimize(init, 300, 1e-9, 1e-12, True, False, None, None, spec)

E_fit = np.sort(fit.obs_micro_np.sum(axis=2)[:, 1])
print(f"HMM states: E = {E_fit.round(3)}   truth {E_STATES}")

## Plot



In [ ]:
fig, (ax_hist, ax_states) = plt.subplots(1, 2, figsize=(10.5, 4.0))

ax_hist.hist(E_burst, bins=np.linspace(0, 1, 41), color="#9aa4ad",
             edgecolor="white", linewidth=0.5)
for E, c in zip(E_STATES, STATE_COLORS):
    ax_hist.axvline(E, color=c, lw=2, ls="--")
ax_hist.set_xlabel("burst-wise FRET efficiency")
ax_hist.set_ylabel("bursts")
ax_hist.set_title("Burst-wise: one peak\ndashed: the three true states",
                  fontsize=10, loc="left")
ax_hist.grid(alpha=0.25, lw=0.6, axis="y")

x = np.arange(3)
ax_states.bar(x - 0.18, E_STATES, width=0.34, color="#9aa4ad", label="truth")
ax_states.bar(x + 0.18, E_fit, width=0.34,
              color=STATE_COLORS, label="HMM", alpha=0.9)
ax_states.set_xticks(x)
ax_states.set_xticklabels([f"state {i}" for i in range(3)])
ax_states.set_ylabel("FRET efficiency")
ax_states.set_ylim(0, 1)
ax_states.set_title("Photon-wise: three states recovered\nsame bursts, same photons",
                    fontsize=10, loc="left")
ax_states.legend(frameon=False, fontsize=9)
ax_states.grid(alpha=0.25, lw=0.6, axis="y")

fig.tight_layout()
plt.show()

## What this shows
The burst histogram is narrow and centred — a textbook "single dynamic
population". Nothing about it suggests three states, and no amount of extra
data would change that, because each burst genuinely *is* an average when
exchange outruns diffusion.

The same photons, modelled individually and with their micro-times, separate
into the three states that generated them. That is the case photon-by-photon
HMM exists for, and it is why the pipeline matters: the burst search and
selection above are not preprocessing to be skipped, they are what decides
which photons the model is allowed to explain.

